# 02 — Validated Multi-Temporal SAR & Ground-Truth Training Data Pipeline

Acquires **strictly validated** multi-temporal SAR + topography inputs (**X**) and **cross-sensor validated ground-truth labels** (**y**) for training the 6-class SAR surface classification model (`0: Bare Ground`, `1: Water & Flood`, `2: Tree Lines`, `3: Wetlands`, `4: Cropland`, `5: Urban`).

### Why a Dedicated Training Pipeline (vs. `01_daten_suchen_und_laden.ipynb`)?
1. **Validated Science Products Only (+ Optional Provisional GUNW Coherence):** Keeps `OPERA_L2_RTC-S1_V1`, `NISAR_L2_GCOV_V1`, `OPERA_L3_DSWX-HLS_V1`, and `NASADEM_HGT` (`v001`) strictly validated (`ALLOW_PROVISIONAL_NISAR = False`), while enabling `NISAR_L2_GUNW_PROVISIONAL_V1` (`ALLOW_PROVISIONAL_GUNW = True`, since V1 GUNW is not yet published) as an optional interferometric coherence channel.
2. **Quality & Geometry Masking:** Streams `OPERA_L2_RTC-S1` `*_mask.tif` (excluding radar layover, shadow, and invalid pixels), NISAR `mask` / `numberOfLooks`, and allows valid mountain slopes up to `MAX_SLOPE_DEG` (`60°`) while guarding dynamic water overrides with `MAX_WATER_SLOPE_DEG` (`15°`).
3. **Cross-Sensor Validated Labels (`y`):**
   - **Static Multi-Class Baseline:** **ESA WorldCover 10m v200 (2021)** streamed AOI-only via Cloud-Optimized GeoTIFF range requests and mapped to project classes `0..5`.
   - **Dynamic Water / Flood Validation:** **OPERA L3 `DSWx-HLS` (`OPERA_L3_DSWX-HLS_V1`)** (optical HLS-derived, independent from SAR!) filtered to cloud/snow-free high-confidence pixels and temporally matched (`±DSWX_MAX_TIME_DIFF_DAYS`) to update seasonal water and inundated vegetation labels.
   - **Patch Homogeneity Filter:** Optionally keeps only pixels whose `3x3` label neighborhood is uniform (`REQUIRE_HOMOGENEOUS_3X3 = True`), avoiding mixed-boundary label noise.
4. **Leakage-Free Spatial-Block Split & Stratified Export:** Splits each AOI on a `BLOCK_SIZE_M` spatial grid into disjoint `train` / `val` / `test` sets across all timestamps, then exports class-balanced `.npz`, `.parquet`, NetCDF-4 cubes, and a validation manifest under `data/training/<RUN_NAME>/`.


## 1. Setup, Dependencies & Earthdata Authentication

In [ ]:
import gc
import hashlib
import json
import math
import os
import re
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import earthaccess
import geopandas as gpd
import h5py
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.features
import rioxarray  # noqa: F401 (registers .rio accessor)
import xarray as xr
from rasterio.enums import Resampling
from rioxarray.exceptions import NoDataInBounds
from scipy.ndimage import binary_dilation, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import Polygon, box
from shapely.ops import unary_union

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), Path.cwd().parent] if (p / ".env").exists()),
    Path.cwd(),
)
ENV_PATH = PROJECT_ROOT / ".env"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAINING_DIR = PROJECT_ROOT / "data" / "training"
RAW_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

if not os.environ.get("EARTHDATA_TOKEN"):
    raise ValueError("EARTHDATA_TOKEN not found. Please set it in .env.")

auth = earthaccess.login(strategy="environment")
print("Earthdata authenticated:", auth.authenticated)
print("Raw cache dir:          ", RAW_DIR)
print("Training output dir:    ", TRAINING_DIR)


## 2. Configuration: Run Profiles (`test` vs `overnight`), AOIs & Sampling

Switch `RUN_PROFILE` between:
* **`"test"`** — Fast end-to-end smoke test: 3 compact AOIs covering all 6 classes, high timestamp separation (`MAX_TIMESTEPS_PER_AOI = 5`, ~5.5 months apart across `2025-2026`), small caps (`4,000 / 1,000 / 1,000` per class per split), and full NetCDF-4 `.nc` U-TAE cubes.
* **`"overnight"`** — Full overnight dataset tuned for 100 GB disk / 32 GB RAM / RTX 4070 Super (12 GB VRAM): all 9 compact multi-biome AOIs, **no timestamp separation** (`MAX_TIMESTEPS_PER_AOI = None`, every valid acquisition) for tabular exports, date-batched parallel streaming with per-AOI parquet caching (`1,500 / 500 / 500` per-timestamp cap; `250,000 / 50,000 / 50,000` AOI-balanced export caps), plus NetCDF-4 `.nc` U-TAE cubes (`MAX_CUBE_TIMESTEPS_PER_AOI = 16` for OPERA RTC-S1 and all valid GCOV/GUNW/DSWx/static/label dates).


In [ ]:
# --- Run Profile Selector ("test" | "overnight") ---
RUN_PROFILE = "overnight"

# Compact single-tile AOIs (west, south, east, north) in EPSG:4326 (~15-20 km, 1 NASADEM + 1 WorldCover tile each)
ALL_TRAINING_AOIS = {
    "oberrhein": (8.33, 49.05, 8.51, 49.21),          # Temperate valley (DE): forest, cropland, river, urban
    "alps_inntal": (11.30, 47.22, 11.48, 47.38),      # Alpine (AT): steep bare rock, mountain forest, valley urban
    "danube_delta": (29.10, 45.02, 29.28, 45.18),     # Delta wetlands (RO): reedbeds, channels, seasonal flood
    "po_valley": (11.48, 44.80, 11.66, 44.96),        # Floodplain (IT): intensive cropland, Ferrara urban, river
    "mekong_can_tho": (105.70, 10.02, 105.86, 10.18), # Monsoon delta (VN): flooded rice paddies, Hau river, urban
    "atchafalaya_la": (-91.50, 29.38, -91.32, 29.54), # Subtropical basin (US): forested swamp, coastal marsh, water
    "nile_cairo": (31.15, 30.02, 31.33, 30.18),       # Hyper-arid (EG): flat desert bare ground, mega-city, Nile, crops
    "pantanal_br": (-58.68, -27.48, -58.50, -27.32),  # Southern Hemisphere floodplain (AR/PY): seasonal wetland, savanna, gallery forest
    "phoenix_az": (-112.22, 33.36, -112.04, 33.52),   # Arid Southwest (US): desert scrub, sprawl urban, irrigated ag
}

PROFILE_CONFIGS = {
    "test": {
        "run_name": "validated_train_test_v1",
        "aoi_keys": ["oberrhein", "alps_inntal", "danube_delta"],
        # High timestamp separation (~5.5 months between evenly spaced dates across 2025-2026) while keeping T > 4
        "max_timesteps_per_aoi": 5,
        "max_cube_timesteps_per_aoi": None,
        "max_samples_per_class_per_timestamp": {"train": 1000, "val": 300, "test": 300},
        "max_samples_per_class_per_split": {"train": 4000, "val": 1000, "test": 1000},
        "save_intermediate_cubes": True,
    },
    "overnight": {
        "run_name": "validated_train_overnight_v1",
        "aoi_keys": list(ALL_TRAINING_AOIS.keys()),
        # No timestamp separation for tabular export; cap OPERA .nc U-TAE cubes at T=16 evenly spaced dates
        "max_timesteps_per_aoi": None,
        "max_cube_timesteps_per_aoi": 16,
        # Bound intermediate 10m DataFrame RAM (< ~4 GB) while gathering ~300k-900k candidates/class
        "max_samples_per_class_per_timestamp": {"train": 1500, "val": 500, "test": 500},
        # Up to 1.5M train + 300k val + 300k test rows per sensor (~100-150 MB .npz, fits 12 GB VRAM)
        "max_samples_per_class_per_split": {"train": 250000, "val": 50000, "test": 50000},
        "save_intermediate_cubes": True,
    },
}

if RUN_PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}. Choose from {list(PROFILE_CONFIGS)}")

_cfg = PROFILE_CONFIGS[RUN_PROFILE]
RUN_NAME = _cfg["run_name"]
TRAINING_AOIS = {k: ALL_TRAINING_AOIS[k] for k in _cfg["aoi_keys"]}

START_DATE = "2025-01-01"
END_DATE = "2026-11-14"
MAX_TIMESTEPS_PER_AOI = _cfg["max_timesteps_per_aoi"]  # None = use all available dates
MAX_CUBE_TIMESTEPS_PER_AOI = _cfg["max_cube_timesteps_per_aoi"]  # Cap OPERA .nc cube timestamps when overnight
TARGET_RES_M = 10.0            # Shared UTM pixel spacing (m): 10m x 10m pixels
MIN_VALID_PCT = 80.0           # Minimum % valid unmasked pixels to keep a timestamp
MIN_BBOX_COVERAGE_PCT = 85.0   # Minimum % of AOI bbox covered by CMR GPolygons per SAR date
DB_MIN = -40.0                 # Physical minimum clamp (dB) to prevent -140 dB zero-floor artifacts
DB_MAX = 20.0                  # Physical maximum clamp (dB) to prevent extreme corner-reflector spikes

# --- Strict Validation Rules ---
ALLOW_PROVISIONAL_NISAR = False    # Keep False to require strictly validated NISAR GCOV & OPERA V1 products
ALLOW_PROVISIONAL_GUNW = True      # Enable NISAR_L2_GUNW_PROVISIONAL_V1 (V1 GUNW not yet released) as optional coherence
MASK_OPERA_LAYOVER_SHADOW = True   # Use OPERA RTC-S1 *_mask.tif to remove layover/shadow/invalid px
MAX_SLOPE_DEG = 60.0               # Allow valid mountain slopes (up to 60 deg) so model separates steep terrain from Urban; None disables
MAX_WATER_SLOPE_DEG = 15.0         # Ignore DSWx optical water overrides on steep slopes (>15 deg) to prevent terrain-shadow false positives
REQUIRE_HOMOGENEOUS_3X3 = True     # Keep only pixels with uniform 3x3 label neighborhood

# --- Input Sensors & Label Sources ---
INCLUDE_OPERA_RTC_S1 = True        # Validated Sentinel-1 C-band RTC (VV, VH + mask)
INCLUDE_NISAR_GCOV = True          # Validated NISAR L-band GCOV (V1 only unless ALLOW_PROVISIONAL_NISAR=True)
INCLUDE_NISAR_GUNW = True          # NISAR L-band GUNW coherence (provisional V1 if ALLOW_PROVISIONAL_GUNW=True)
INCLUDE_NASADEM = True             # Static terrain (elevation, slope, aspect)

INCLUDE_ESA_WORLDCOVER = True      # Static 10m validated land-cover baseline (public S3 COG)
INCLUDE_DSWX_HLS = True            # Dynamic optical water/flood validation (OPERA_L3_DSWX-HLS_V1)
HIGH_CONFIDENCE_DSWX_ONLY = True   # Keep only DSWx high-confidence water / clear non-water pixels
DSWX_MAX_TIME_DIFF_DAYS = 10       # Max |t_sar - t_dswx| days for dynamic water label matching
GUNW_MAX_TIME_DIFF_DAYS = 12       # Max |t_gcov - t_gunw| days to pair interferometric coherence
VALIDATED_LABELS_GEOJSON = None    # Optional path to vector polygons with integer 'class_id' column

# --- Spatial-Block Split & Sampling ---
BLOCK_SIZE_M = 3000.0              # Spatial block size in meters for leakage-free train/val/test split
SPLIT_RATIOS = (0.70, 0.15, 0.15)  # (train, val, test)
SPLIT_ID_TO_NAME = {0: "train", 1: "val", 2: "test"}
MAX_SAMPLES_PER_CLASS_PER_TIMESTAMP = _cfg["max_samples_per_class_per_timestamp"]
MAX_SAMPLES_PER_CLASS_PER_SPLIT = _cfg["max_samples_per_class_per_split"]
SAVE_INTERMEDIATE_CUBES = _cfg["save_intermediate_cubes"]
RANDOM_SEED = 42
RUN_FETCH = True

# --- Project Surface Class Schema (aligned with src/config.py) ---
IGNORE_LABEL = 255
CLASS_DEFINITIONS = {
    0: ("Bare Ground", "#64748B"),
    1: ("Water & Flood", "#0EA5E9"),
    2: ("Tree Lines & Forest", "#22C55E"),
    3: ("Wetlands", "#14B8A6"),
    4: ("Cropland", "#EAB308"),
    5: ("Urban", "#F97316"),
}

# ESA WorldCover v200 (10..100) -> Project Classes (0..5)
WORLDCOVER_TO_CLASS = {
    10: 2,   # Tree cover -> Tree Lines & Forest
    20: 0,   # Shrubland -> Bare / Sparse Ground
    30: 0,   # Grassland -> Bare / Sparse Ground
    40: 4,   # Cropland -> Cropland
    50: 5,   # Built-up -> Urban
    60: 0,   # Bare / sparse vegetation -> Bare Ground
    70: 0,   # Snow and ice -> Bare / Unclassified (or filtered)
    80: 1,   # Permanent water bodies -> Water & Flood
    90: 3,   # Herbaceous wetland -> Wetlands
    95: 3,   # Mangroves -> Wetlands
    100: 0,  # Moss and lichen -> Bare Ground
}

OUT_ROOT = TRAINING_DIR / RUN_NAME
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Profile: {RUN_PROFILE!r} -> run_name={RUN_NAME!r}")
print(f"Configured {len(TRAINING_AOIS)} AOI(s): {list(TRAINING_AOIS.keys())}")
print(f"Max timesteps/AOI: {MAX_TIMESTEPS_PER_AOI} (OPERA cube cap: {MAX_CUBE_TIMESTEPS_PER_AOI}) | Per-timestamp cap: {MAX_SAMPLES_PER_CLASS_PER_TIMESTAMP}")
print(f"Per-split cap: {MAX_SAMPLES_PER_CLASS_PER_SPLIT} | Save .nc cubes: {SAVE_INTERMEDIATE_CUBES}")
print(f"Strict GCOV/OPERA V1 only: {not ALLOW_PROVISIONAL_NISAR} | Allow GUNW Provisional: {ALLOW_PROVISIONAL_GUNW}")
print(f"Output directory: {OUT_ROOT}")


## 3. Multi-AOI Earthdata Catalog Search & Validation Audit

Queries NASA Earthdata CMR for validated (`V1`) collections only (unless `ALLOW_PROVISIONAL_NISAR=True`). Note that `OPERA_L3_DSWX-HLS_V1` is searched with `±DSWX_MAX_TIME_DIFF_DAYS` padding around `[START_DATE, END_DATE]` so edge SAR acquisitions can still find nearby optical water validation scenes.


In [ ]:
def _extract_granule_date(granule, use_end: bool = False) -> str | None:
    umm = granule.get("umm", {})
    te = umm.get("TemporalExtent", {})
    rdt = te.get("RangeDateTime") or {}
    sdt = te.get("SingleDateTime")
    if use_end:
        raw = rdt.get("EndingDateTime") or rdt.get("BeginningDateTime") or sdt
    else:
        raw = rdt.get("BeginningDateTime") or sdt
    return str(raw)[:10] if raw else None


def _granule_footprints(granule) -> list[Polygon]:
    geom = granule.get("umm", {}).get("SpatialExtent", {}).get("HorizontalSpatialDomain", {}).get("Geometry", {})
    polys: list[Polygon] = []
    for gp in geom.get("GPolygons") or []:
        pts = (gp.get("Boundary") or {}).get("Points") or []
        if len(pts) >= 3:
            coords = [(float(p["Longitude"]), float(p["Latitude"])) for p in pts]
            poly = Polygon(coords).buffer(0)
            if not poly.is_empty and poly.area > 0:
                polys.append(poly)
    if not polys:
        for br in geom.get("BoundingRectangles") or []:
            w = br.get("WestBoundingCoordinate")
            s = br.get("SouthBoundingCoordinate")
            e = br.get("EastBoundingCoordinate")
            n = br.get("NorthBoundingCoordinate")
            if None not in (w, s, e, n):
                polys.append(box(float(w), float(s), float(e), float(n)))
    return polys


def _granule_aoi_coverage_pct(granule, aoi_poly: Polygon) -> float:
    polys = _granule_footprints(granule)
    if not polys:
        return 100.0
    cov = unary_union(polys).intersection(aoi_poly).area / aoi_poly.area * 100.0
    return float(min(100.0, max(0.0, cov)))


def filter_dates_by_aoi_coverage(
    df: pd.DataFrame,
    bbox: tuple[float, float, float, float],
    min_coverage_pct: float,
) -> pd.DataFrame:
    """Retain only dates whose combined granule GPolygons cover >= min_coverage_pct of the AOI bbox."""
    if df.empty or min_coverage_pct <= 0:
        return df
    aoi_poly = box(*bbox)
    keep_dates = set()
    for dt, grp in df.groupby("date"):
        if pd.isna(dt):
            continue
        polys: list[Polygon] = []
        for g in grp["granule"]:
            polys.extend(_granule_footprints(g))
        if not polys:
            keep_dates.add(dt)
            continue
        cov_pct = unary_union(polys).intersection(aoi_poly).area / aoi_poly.area * 100.0
        if cov_pct >= min_coverage_pct:
            keep_dates.add(dt)
    return df[df["date"].isin(keep_dates)].copy()


EXCLUDED_NISAR_POL_MODES = ("_SHSH_", "_NASV_", "_SVSV_")


def search_collection(
    dataset_name: str,
    short_names: list[str],
    bbox: tuple[float, float, float, float],
    temporal: tuple[str, str] | None,
    version: str | None = None,
) -> list[dict]:
    records = []
    seen = set()
    aoi_poly = box(*bbox)
    use_end_dt = dataset_name == "nisar_gunw"
    for sn in short_names:
        kwargs = {"short_name": sn, "bounding_box": bbox}
        if temporal is not None:
            kwargs["temporal"] = temporal
        if version is not None:
            kwargs["version"] = version

        granules = earthaccess.search_data(**kwargs)
        for g in granules:
            native_id = g.get("meta", {}).get("native-id", "")
            if dataset_name == "nisar_gcov" and any(tok in native_id.upper() for tok in EXCLUDED_NISAR_POL_MODES):
                continue
            dedup_key = native_id.replace("_PROVISIONAL_", "_")
            if dedup_key in seen:
                continue
            seen.add(dedup_key)
            raw_size = g.size() if callable(getattr(g, "size", None)) else g.get("size", 0.0)
            is_provisional = "PROVISIONAL" in sn.upper() or "_PROVISIONAL_" in native_id.upper()
            records.append(
                {
                    "dataset": dataset_name,
                    "short_name": sn,
                    "granule_id": native_id,
                    "date": _extract_granule_date(g, use_end=use_end_dt),
                    "aoi_cov_pct": round(_granule_aoi_coverage_pct(g, aoi_poly), 2),
                    "size_mb": float(raw_size or 0.0),
                    "validated_product": not is_provisional,
                    "granule": g,
                }
            )
    return records


def select_dates_evenly(df: pd.DataFrame, max_dates: int | None) -> pd.DataFrame:
    if df.empty or max_dates is None:
        return df
    unique_dates = sorted(d for d in df["date"].dropna().unique())
    if len(unique_dates) <= max_dates:
        return df
    idx = np.linspace(0, len(unique_dates) - 1, max_dates, dtype=int)
    chosen = {unique_dates[i] for i in idx}
    return df[df["date"].isin(chosen)].copy()


def deduplicate_dswx_reprocessing(dswx_df: pd.DataFrame) -> pd.DataFrame:
    """Keep only the latest processing timestamp for each DSWx-HLS tile/acquisition/sensor."""
    if dswx_df.empty:
        return dswx_df
    pat = re.compile(r"^(OPERA_L3_DSWx-HLS_[^_]+_\d{8}T\d{6}Z)_(\d{8}T\d{6}Z)_(.+)$")
    acq_keys = []
    prod_ts = []
    for idx, gid in enumerate(dswx_df["granule_id"].astype(str)):
        m = pat.match(gid)
        if m:
            acq_keys.append(f"{m.group(1)}_{m.group(3)}")
            prod_ts.append(m.group(2))
        else:
            acq_keys.append(f"__raw_{idx}_{gid}")
            prod_ts.append("")
    work = dswx_df.copy()
    work["_acq_key"] = acq_keys
    work["_prod_ts"] = prod_ts
    work = (
        work.sort_values(["_acq_key", "_prod_ts"], kind="mergesort")
        .drop_duplicates(subset=["_acq_key"], keep="last")
        .drop(columns=["_acq_key", "_prod_ts"])
        .sort_index()
    )
    return work


def filter_dswx_near_sar_dates(
    dswx_df: pd.DataFrame, sar_dates: list[str] | set[str], max_diff_days: int
) -> pd.DataFrame:
    """Retain only granules within +/- max_diff_days of any reference SAR date."""
    if dswx_df.empty:
        return dswx_df
    sar_ts = pd.to_datetime(sorted(set(sar_dates)), errors="coerce").dropna()
    if len(sar_ts) == 0:
        return dswx_df.iloc[0:0].copy()
    dswx_ts = pd.to_datetime(dswx_df["date"], errors="coerce")
    sar_ns = sar_ts.normalize().to_numpy(dtype="datetime64[D]")
    dswx_ns = dswx_ts.dt.normalize().to_numpy(dtype="datetime64[D]")
    valid = ~np.isnat(dswx_ns)
    keep_mask = np.zeros(len(dswx_df), dtype=bool)
    if valid.any():
        diffs = np.abs((dswx_ns[valid, None] - sar_ns[None, :]).astype("timedelta64[D]").astype(np.int64))
        keep_mask[valid] = diffs.min(axis=1) <= int(max_diff_days)
    return dswx_df.loc[keep_mask].copy()


dswx_start = str((pd.Timestamp(START_DATE) - pd.Timedelta(days=DSWX_MAX_TIME_DIFF_DAYS)).date())
dswx_end = str((pd.Timestamp(END_DATE) + pd.Timedelta(days=DSWX_MAX_TIME_DIFF_DAYS)).date())

nisar_gcov_sns = ["NISAR_L2_GCOV_V1"] + (["NISAR_L2_GCOV_PROVISIONAL_V1"] if ALLOW_PROVISIONAL_NISAR else [])
nisar_gunw_sns = ["NISAR_L2_GUNW_V1"] + (
    ["NISAR_L2_GUNW_PROVISIONAL_V1"] if (ALLOW_PROVISIONAL_NISAR or ALLOW_PROVISIONAL_GUNW) else []
)

catalog_by_aoi: dict[str, pd.DataFrame] = {}
selected_by_aoi: dict[str, pd.DataFrame] = {}

for aoi_name, bbox in TRAINING_AOIS.items():
    plan = []
    if INCLUDE_OPERA_RTC_S1:
        plan.append(("opera_rtc_s1", ["OPERA_L2_RTC-S1_V1"], (START_DATE, END_DATE), None))
    if INCLUDE_NISAR_GCOV:
        plan.append(("nisar_gcov", nisar_gcov_sns, (START_DATE, END_DATE), None))
    if INCLUDE_NISAR_GUNW:
        plan.append(("nisar_gunw", nisar_gunw_sns, (START_DATE, END_DATE), None))
    if INCLUDE_DSWX_HLS:
        plan.append(("opera_dswx_hls", ["OPERA_L3_DSWX-HLS_V1"], (dswx_start, dswx_end), None))
    if INCLUDE_NASADEM:
        plan.append(("nasadem", ["NASADEM_HGT"], None, "001"))

    recs = []
    for ds_name, sns, temp, ver in plan:
        recs.extend(search_collection(ds_name, sns, bbox, temp, version=ver))

    cat_df = pd.DataFrame(recs)
    catalog_by_aoi[aoi_name] = cat_df
    if cat_df.empty:
        print(f"[WARN] {aoi_name}: No granules found!")
        selected_by_aoi[aoi_name] = cat_df
        continue

    sel_parts = []
    selected_sar_dates: set[str] = set()
    selected_gcov_dates: set[str] = set()
    for ds_name, grp in cat_df.groupby("dataset"):
        if ds_name in ("opera_dswx_hls", "nisar_gunw"):
            continue
        if ds_name == "nasadem":
            sel_ds = grp
        else:
            grp_cov = filter_dates_by_aoi_coverage(grp, bbox, MIN_BBOX_COVERAGE_PCT)
            n_raw_d, n_cov_d = grp["date"].nunique(), grp_cov["date"].nunique()
            if n_cov_d < n_raw_d:
                print(
                    f"[{aoi_name}] {ds_name} AOI coverage pre-filter (>={MIN_BBOX_COVERAGE_PCT:.0f}%): "
                    f"{n_raw_d} -> {n_cov_d} dates ({len(grp)} -> {len(grp_cov)} granules)"
                )
            sel_ds = select_dates_evenly(grp_cov, MAX_TIMESTEPS_PER_AOI)
            if ds_name in ("opera_rtc_s1", "nisar_gcov"):
                ds_dates = {d for d in sel_ds["date"].dropna().astype(str)}
                selected_sar_dates.update(ds_dates)
                if ds_name == "nisar_gcov":
                    selected_gcov_dates.update(ds_dates)
        sel_parts.append(sel_ds)

    gunw_raw = cat_df[cat_df["dataset"] == "nisar_gunw"]
    if not gunw_raw.empty:
        gunw_cov = filter_dates_by_aoi_coverage(gunw_raw, bbox, MIN_BBOX_COVERAGE_PCT)
        gunw_filt = filter_dswx_near_sar_dates(gunw_cov, selected_gcov_dates, GUNW_MAX_TIME_DIFF_DAYS)
        n_gunw_dates = gunw_filt["date"].nunique() if not gunw_filt.empty else 0
        print(
            f"[{aoi_name}] NISAR GUNW pre-filter (cov>={MIN_BBOX_COVERAGE_PCT:.0f}%, +/-{GUNW_MAX_TIME_DIFF_DAYS}d of {len(selected_gcov_dates)} GCOV dates): "
            f"{len(gunw_raw)} -> {len(gunw_cov)} -> {len(gunw_filt)} granules ({n_gunw_dates} unique dates)"
        )
        sel_parts.append(gunw_filt)

    dswx_raw = cat_df[cat_df["dataset"] == "opera_dswx_hls"]
    if not dswx_raw.empty:
        dswx_dedup = deduplicate_dswx_reprocessing(dswx_raw)
        dswx_filt = filter_dswx_near_sar_dates(dswx_dedup, selected_sar_dates, DSWX_MAX_TIME_DIFF_DAYS)
        n_dswx_dates = dswx_filt["date"].nunique() if not dswx_filt.empty else 0
        print(
            f"[{aoi_name}] DSWx-HLS pre-filter (+/-{DSWX_MAX_TIME_DIFF_DAYS}d of {len(selected_sar_dates)} SAR dates): "
            f"{len(dswx_raw)} -> {len(dswx_dedup)} (dedup) -> {len(dswx_filt)} granules ({n_dswx_dates} unique dates)"
        )
        sel_parts.append(dswx_filt)

    sel_df = pd.concat(sel_parts, ignore_index=True) if sel_parts else cat_df.iloc[0:0].copy()
    selected_by_aoi[aoi_name] = sel_df

    sel_counts = sel_df.groupby("dataset")["date"].nunique().to_dict() if not sel_df.empty else {}
    audit = (
        cat_df.groupby("dataset", as_index=False)
        .agg(
            total_granules=("granule_id", "count"),
            unique_dates=("date", "nunique"),
            all_validated=("validated_product", "all"),
            first_date=("date", "min"),
            last_date=("date", "max"),
        )
        .sort_values("dataset")
    )
    audit["selected_dates"] = audit["dataset"].map(sel_counts).fillna(0).astype(int)
    print(f"\n=== Catalog Audit: {aoi_name} ===")
    display(audit)
    if INCLUDE_NISAR_GCOV and "nisar_gcov" not in set(cat_df["dataset"]):
        print(
            f"[INFO] {aoi_name}: No strictly validated NISAR_L2_GCOV_V1 granules found in {START_DATE}..{END_DATE}. "
            "OPERA_L2_RTC-S1_V1 will still be processed (or set ALLOW_PROVISIONAL_NISAR=True if needed)."
        )


## 4. Remote AOI-Only Input Fetchers with Quality & Layover/Shadow Masking

All SAR layers are streamed via `earthaccess.open()` and clipped to the padded `AOI_BBOX` before reprojection onto each AOI's UTM `TARGET_RES_M` grid:
* **OPERA `RTC-S1`:** Pairs each backscatter COG (`_VV.tif`, `_VH.tif`, `_HH.tif`, `_HV.tif`) with its granule's `*_mask.tif` and masks out any pixel with `mask != 0` (layover, shadow, invalid).
* **NISAR `GCOV`:** Reads `/science/LSAR/GCOV/grids/frequencyA` (or `frequencyB`) covariance terms plus `mask` and `numberOfLooks` when present, masking invalid or zero-look pixels.


In [ ]:
def build_target_grid(aoi_wgs84: gpd.GeoDataFrame, target_crs, res_m: float) -> xr.DataArray:
    aoi_utm = aoi_wgs84.to_crs(target_crs)
    minx, miny, maxx, maxy = aoi_utm.total_bounds
    minx = np.floor(minx / res_m) * res_m
    miny = np.floor(miny / res_m) * res_m
    maxx = np.ceil(maxx / res_m) * res_m
    maxy = np.ceil(maxy / res_m) * res_m

    xs = np.arange(minx + res_m / 2, maxx, res_m)
    ys = np.arange(maxy - res_m / 2, miny, -res_m)

    grid = xr.DataArray(
        np.zeros((len(ys), len(xs)), dtype=np.float32),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    grid = grid.rio.write_crs(target_crs).rio.set_spatial_dims(x_dim="x", y_dim="y")
    return grid


def align_to_target(da: xr.DataArray, target_grid: xr.DataArray, resampling=Resampling.bilinear) -> xr.DataArray:
    if da.rio.crs is None:
        raise ValueError("DataArray is missing CRS.")
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")
    return da.rio.reproject_match(target_grid, resampling=resampling)


def mosaic_aligned(parts: list[xr.DataArray], mode: str = "mean") -> xr.DataArray:
    if len(parts) == 1:
        return parts[0].astype("float32")
    if mode == "first":
        out = parts[0]
        for nxt in parts[1:]:
            out = out.combine_first(nxt)
        return out.astype("float32")
    stacked = xr.concat(parts, dim="_mosaic")
    return stacked.mean(dim="_mosaic", skipna=True).astype("float32")


def to_db(da: xr.DataArray) -> xr.DataArray:
    valid = da.where(da > 0)
    db = (10.0 * np.log10(valid)).clip(min=DB_MIN, max=DB_MAX)
    return db.where(np.isfinite(valid)).astype("float32")


def _resolve_earthdata_url(granule, url: str) -> str:
    if "datapool.asf.alaska.edu" not in url:
        return url
    direct_by_name = {u.rsplit("/", 1)[-1]: u for u in granule.data_links(access="direct")}
    fname = url.rsplit("/", 1)[-1]
    s3_url = direct_by_name.get(fname, "")
    if s3_url.startswith("s3://asf-cumulus-prod-opera-products/"):
        return s3_url.replace(
            "s3://asf-cumulus-prod-opera-products/",
            "https://cumulus.asf.earthdatacloud.nasa.gov/OPERA/",
            1,
        )
    return url


def _filter_granule_urls(
    granules: list, suffixes: tuple[str, ...], use_end_date: bool = False
) -> list[tuple[str, str]]:
    items = []
    for g in granules:
        dt = _extract_granule_date(g, use_end=use_end_date)
        for url in g.data_links(access="external"):
            if url.endswith(suffixes):
                items.append((dt, _resolve_earthdata_url(g, url)))
    return items


def _safe_earthaccess_open(urls: list[str], show_progress: bool = True, max_retries: int = 2) -> list:
    """Open remote Earthdata URLs via fsspec with retry and clean 404/error skipping."""
    if not urls:
        return []
    results = [None] * len(urls)
    pending = list(range(len(urls)))
    for attempt in range(max_retries + 1):
        batch_urls = [urls[i] for i in pending]
        try:
            opened = earthaccess.open(
                batch_urls,
                show_progress=show_progress and attempt == 0,
                pqdm_kwargs={"exception_behaviour": "ignore", "n_jobs": 8},
            )
        except Exception as exc:
            print(f"  [WARN] earthaccess.open attempt {attempt + 1} failed: {exc}")
            opened = [exc] * len(batch_urls)
        still_pending = []
        for idx, f in zip(pending, opened):
            if isinstance(f, BaseException) or not hasattr(f, "read"):
                still_pending.append(idx)
            else:
                results[idx] = f
        pending = still_pending
        if not pending:
            break
        if attempt < max_retries:
            time.sleep(2 * (attempt + 1))
    if pending:
        print(f"  [WARN] Skipped {len(pending)}/{len(urls)} remote file(s) after {max_retries + 1} attempt(s).")
    return results


def _clip_remote_cog(
    file_obj,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
    resampling=Resampling.bilinear,
    masked: bool = True,
) -> xr.DataArray | None:
    try:
        with rioxarray.open_rasterio(file_obj, masked=masked) as da:
            da = da.squeeze("band", drop=True)
            if da.rio.crs is None:
                return None
            minx, miny, maxx, maxy = aoi_wgs84.to_crs(da.rio.crs).total_bounds
            pad = max(TARGET_RES_M * 4.0, 120.0)
            try:
                clipped = da.rio.clip_box(minx - pad, miny - pad, maxx + pad, maxy + pad)
            except NoDataInBounds:
                return None
            if clipped.size == 0:
                return None
            return align_to_target(clipped, target_grid, resampling=resampling).load()
    finally:
        if hasattr(file_obj, "close"):
            try:
                file_obj.close()
            except Exception:
                pass


def _stream_and_clip_cogs(
    url_items: list[tuple],
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
    chunk_size: int = 128,
    max_workers: int = 8,
    show_progress: bool = True,
) -> dict:
    """Open Earthdata COG URLs in bounded chunks and clip in parallel with fault tolerance."""
    out: dict = {}
    if not url_items:
        return out

    def _clip_one(args):
        key, fobj, resampling, masked = args
        try:
            return key, _clip_remote_cog(
                fobj, aoi_wgs84, target_grid, resampling=resampling, masked=masked
            )
        except Exception as exc:
            print(f"  [WARN] COG read/clip failed for {key}: {exc}")
            if hasattr(fobj, "close"):
                try:
                    fobj.close()
                except Exception:
                    pass
            return key, None

    for start in range(0, len(url_items), chunk_size):
        chunk = url_items[start : start + chunk_size]
        open_files = _safe_earthaccess_open(
            [u for _, u, _, _ in chunk], show_progress=show_progress
        )
        tasks = [
            (key, fobj, resampling, masked)
            for (key, _, resampling, masked), fobj in zip(chunk, open_files)
            if fobj is not None
        ]
        if not tasks:
            continue
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            for key, clipped in pool.map(_clip_one, tasks):
                if clipped is not None:
                    out[key] = clipped
    return out


def _read_nisar_epsg(group: h5py.Group) -> int:
    proj = group["projection"]
    epsg_attr = proj.attrs.get("epsg_code")
    if epsg_attr is not None:
        return int(epsg_attr)
    return int(proj[()])


def _slice_h5_datasets(
    grp: h5py.Group,
    dataset_paths: list[str],
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
    resampling=Resampling.bilinear,
) -> dict[str, xr.DataArray]:
    epsg = _read_nisar_epsg(grp)
    xs = grp["xCoordinates"][:]
    ys = grp["yCoordinates"][:]

    minx, miny, maxx, maxy = aoi_wgs84.to_crs(epsg=epsg).total_bounds
    pad = max(TARGET_RES_M * 4.0, 120.0)
    xi = np.where((xs >= minx - pad) & (xs <= maxx + pad))[0]
    yi = np.where((ys >= miny - pad) & (ys <= maxy + pad))[0]
    if len(xi) == 0 or len(yi) == 0:
        return {}

    x0, x1 = int(xi[0]), int(xi[-1]) + 1
    y0, y1 = int(yi[0]), int(yi[-1]) + 1
    sub_x = xs[x0:x1].astype("float64")
    sub_y = ys[y0:y1].astype("float64")

    out = {}
    for dpath in dataset_paths:
        if dpath not in grp:
            continue
        raw_arr = grp[dpath][y0:y1, x0:x1]
        if dpath == "mask" and "unwrappedInterferogram" in grp.name:
            m_u32 = raw_arr.astype(np.uint64)
            low_b = m_u32 & 0xFF
            gunw_ok = (
                (m_u32 != 255)
                & ((low_b % 10) > 0)
                & (((low_b // 10) % 10) > 0)
                & (((m_u32 >> 8) & 0xFFFF) == 0)
            )
            arr = gunw_ok.astype("float32")
        else:
            arr = raw_arr.astype("float32")
        da = xr.DataArray(arr, coords={"y": sub_y, "x": sub_x}, dims=("y", "x"))
        da = da.rio.write_crs(f"EPSG:{epsg}").rio.set_spatial_dims(x_dim="x", y_dim="y")
        rs = Resampling.nearest if dpath.endswith("mask") else resampling
        out[dpath] = align_to_target(da, target_grid, resampling=rs)
    return out


def fetch_nasadem_static(
    aoi_name: str,
    nasadem_df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if nasadem_df.empty:
        return None
    ds_dir = RAW_DIR / f"{RUN_NAME}_{aoi_name}" / "nasadem"
    ds_dir.mkdir(parents=True, exist_ok=True)
    paths = earthaccess.download(nasadem_df["granule"].tolist(), local_path=str(ds_dir))

    hgt_files = []
    for p in paths:
        p = Path(p)
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p, "r") as zf:
                zf.extractall(ds_dir)
                hgt_files.extend(ds_dir / m for m in zf.namelist() if m.lower().endswith(".hgt"))
        elif p.suffix.lower() == ".hgt":
            hgt_files.append(p)

    parts = []
    for p in hgt_files:
        da = rioxarray.open_rasterio(p, masked=True).squeeze("band", drop=True)
        if da.rio.crs is None:
            da = da.rio.write_crs("EPSG:4326")
        minx, miny, maxx, maxy = aoi_wgs84.total_bounds
        da = da.rio.clip_box(minx - 0.01, miny - 0.01, maxx + 0.01, maxy + 0.01)
        parts.append(align_to_target(da, target_grid, resampling=Resampling.bilinear))

    if not parts:
        return None
    elev = mosaic_aligned(parts, mode="mean")
    dy_dx = np.gradient(elev.values, TARGET_RES_M, TARGET_RES_M)
    dz_dy, dz_dx = -dy_dx[0], dy_dx[1]
    slope_deg = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype("float32")
    aspect_deg = (np.degrees(np.arctan2(dz_dy, -dz_dx)) % 360.0).astype("float32")
    aspect_rad = np.radians(aspect_deg)

    ds_static = xr.Dataset(
        {
            "elevation_m": elev,
            "slope_deg": elev.copy(data=slope_deg),
            "aspect_deg": elev.copy(data=aspect_deg),
            "aspect_sin": elev.copy(data=np.sin(aspect_rad).astype("float32")),
            "aspect_cos": elev.copy(data=np.cos(aspect_rad).astype("float32")),
        }
    )
    return ds_static.rio.write_crs(target_grid.rio.crs)


def fetch_opera_rtc_validated(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None

    pol_suffixes = ("_VV.tif", "_VH.tif", "_HH.tif", "_HV.tif")
    granule_entries = []
    urls_to_open = []
    url_roles = []  # (entry_idx, role)

    for g in df["granule"].tolist():
        dt = _extract_granule_date(g)
        pol_urls = {}
        mask_url = None
        for u in g.data_links(access="external"):
            resolved = _resolve_earthdata_url(g, u)
            if resolved.endswith(pol_suffixes):
                pol = resolved.rsplit("_", 1)[-1].replace(".tif", "")
                pol_urls[pol] = resolved
            elif MASK_OPERA_LAYOVER_SHADOW and resolved.endswith("_mask.tif"):
                mask_url = resolved
        if not pol_urls:
            continue
        idx = len(granule_entries)
        granule_entries.append({"date": dt, "pol_urls": pol_urls, "mask_url": mask_url})
        if mask_url is not None:
            urls_to_open.append((("mask", idx), mask_url, Resampling.nearest, False))
        for pol, pu in pol_urls.items():
            urls_to_open.append((("pol", idx, pol), pu, Resampling.bilinear, True))

    if not urls_to_open:
        return None

    print(f"OPERA RTC-S1 (validated): streaming {len(urls_to_open)} COG files (incl. masks)...")
    clipped_map = _stream_and_clip_cogs(
        urls_to_open, aoi_wgs84, target_grid, chunk_size=128, max_workers=8, show_progress=True
    )

    by_date_pol: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))
    for entry_idx, entry in enumerate(granule_entries):
        dt = entry["date"]
        valid_mask = None
        m_da = clipped_map.pop(("mask", entry_idx), None)
        if m_da is not None:
            # OPERA RTC-S1 mask: 0 = valid; non-zero = layover/shadow/invalid
            valid_mask = m_da == 0

        for pol in entry["pol_urls"]:
            da = clipped_map.pop(("pol", entry_idx, pol), None)
            if da is None:
                continue
            if valid_mask is not None:
                da = da.where(valid_mask)
            by_date_pol[dt][pol].append(da)

    if not by_date_pol:
        return None
    dates = sorted(by_date_pol.keys())
    all_pols = sorted({pol for pmap in by_date_pol.values() for pol in pmap})
    ds_dict = {}
    for pol in all_pols:
        per_date = [
            to_db(mosaic_aligned(by_date_pol[dt][pol], mode="mean"))
            if by_date_pol[dt].get(pol)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"{pol}_dB"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))

    ds = xr.Dataset(ds_dict)
    if "VH_dB" in ds and "VV_dB" in ds:
        ds["VH_minus_VV_dB"] = (ds["VH_dB"] - ds["VV_dB"]).astype("float32")
    if "HV_dB" in ds and "HH_dB" in ds:
        ds["HH_minus_HV_dB"] = (ds["HH_dB"] - ds["HV_dB"]).astype("float32")
    return ds.rio.write_crs(target_grid.rio.crs)


def fetch_nisar_gcov_validated(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), (".h5",))
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"NISAR GCOV: streaming {len(urls)} HDF5 AOI slices...")
    open_files = _safe_earthaccess_open(urls, show_progress=True)

    pol_terms = ["HHHH", "HVHV", "VVVV", "VHVH"]
    extra_layers = ["mask", "numberOfLooks"]
    by_date_term: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))

    for (dt, _), fobj in zip(items, open_files):
        if fobj is None:
            continue
        try:
            with h5py.File(fobj, "r") as hf:
                freq_path = "/science/LSAR/GCOV/grids/frequencyA"
                if freq_path not in hf:
                    freq_path = "/science/LSAR/GCOV/grids/frequencyB"
                if freq_path not in hf:
                    continue
                sliced = _slice_h5_datasets(hf[freq_path], pol_terms + extra_layers, aoi_wgs84, target_grid)
                q_mask = None
                if "mask" in sliced:
                    m = sliced["mask"]
                    q_mask = np.isfinite(m) & (m > 0) & (m < 255)
                if "numberOfLooks" in sliced:
                    looks = sliced["numberOfLooks"]
                    looks_ok = np.isfinite(looks) & (looks > 0)
                    q_mask = looks_ok if q_mask is None else (q_mask & looks_ok)

                for term in pol_terms:
                    if term in sliced:
                        da = sliced[term]
                        if q_mask is not None:
                            da = da.where(q_mask)
                        by_date_term[dt][term].append(da)
        except Exception as exc:
            print(f"  [WARN] Skipped NISAR GCOV HDF5 ({dt}): {exc}")
        finally:
            if hasattr(fobj, "close"):
                try:
                    fobj.close()
                except Exception:
                    pass

    if not by_date_term:
        return None
    dates = sorted(by_date_term.keys())
    present_terms = sorted({t for tmap in by_date_term.values() for t in tmap})
    ds_dict = {}
    for term in present_terms:
        per_date = [
            to_db(mosaic_aligned(by_date_term[dt][term], mode="mean"))
            if by_date_term[dt].get(term)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"{term}_dB"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))

    ds = xr.Dataset(ds_dict)
    if "HHHH_dB" in ds and "HVHV_dB" in ds:
        ds["HH_minus_HV_dB"] = (ds["HHHH_dB"] - ds["HVHV_dB"]).astype("float32")
    return ds.rio.write_crs(target_grid.rio.crs)


def fetch_nisar_gunw_validated(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), (".h5",), use_end_date=True)
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"NISAR GUNW: streaming {len(urls)} HDF5 AOI slices...")
    open_files = _safe_earthaccess_open(urls, show_progress=True)

    coh_paths = [f"{pol}/coherenceMagnitude" for pol in ("HH", "VV", "HV", "VH")]
    by_date_pol: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))

    for (dt, _), fobj in zip(items, open_files):
        if fobj is None:
            continue
        try:
            with h5py.File(fobj, "r") as hf:
                base = "/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram"
                if base not in hf:
                    base = "/science/LSAR/GUNW/grids/frequencyB/unwrappedInterferogram"
                if base not in hf:
                    continue
                sliced = _slice_h5_datasets(hf[base], coh_paths + ["mask"], aoi_wgs84, target_grid)
                q_mask = None
                if "mask" in sliced:
                    m = sliced["mask"]
                    q_mask = np.isfinite(m) & (m >= 0.5)
                for cpath, da in sliced.items():
                    if cpath == "mask":
                        continue
                    pol = cpath.split("/", 1)[0]
                    clean_da = da.where((da >= 0.0) & (da <= 1.0))
                    if q_mask is not None:
                        clean_da = clean_da.where(q_mask)
                    by_date_pol[dt][pol].append(clean_da.astype("float32"))
        except Exception as exc:
            print(f"  [WARN] Skipped NISAR GUNW HDF5 ({dt}): {exc}")
        finally:
            if hasattr(fobj, "close"):
                try:
                    fobj.close()
                except Exception:
                    pass

    if not by_date_pol:
        return None
    dates = sorted(by_date_pol.keys())
    present_pols = sorted({pol for pmap in by_date_pol.values() for pol in pmap})
    ds_dict = {}
    for pol in present_pols:
        per_date = [
            mosaic_aligned(by_date_pol[dt][pol], mode="mean").clip(min=0.0, max=1.0)
            if by_date_pol[dt].get(pol)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"coherence_{pol}"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))
    return xr.Dataset(ds_dict).rio.write_crs(target_grid.rio.crs)


## 5. Cross-Sensor Validated Ground-Truth Label Fetchers (`y`)

Combines:
1. **ESA WorldCover 10m v200 (2021)** — Streamed AOI-only from public S3 COGs (`3° x 3°` tiles), nearest-neighbor aligned to `target_grid`, and mapped to `0..5`. With `REQUIRE_HOMOGENEOUS_3X3=True`, pixels on mixed class boundaries are masked to `255` (`IGNORE_LABEL`).
2. **OPERA L3 `DSWx-HLS` (`OPERA_L3_DSWX-HLS_V1`)** — Independent optical water classification (`_B01_WTR.tif` + `_B03_CONF.tif`). Clouds (`253`), snow/ice (`252`), ocean (`254`), and fill (`255`) are excluded. Within `±DSWX_MAX_TIME_DIFF_DAYS` of each SAR date:
   * `DSWx == 1` (open water) -> class `1` (`Water & Flood`)
   * `DSWx == 2` (partial surface water / flooded vegetation) over vegetated/cropland terrain -> class `3` (`Wetlands`)
   * `DSWx == 0` (clear non-water) over static WorldCover water (`1`) -> marks stale water pixels as `255` (`IGNORE_LABEL`) so dried-up waterbodies don't pollute training!
3. **Optional Custom Vector Polygons (`VALIDATED_LABELS_GEOJSON`)** — Overrides raster labels if supplied.


In [ ]:
def worldcover_tiles_for_bbox(bbox: tuple[float, float, float, float]) -> list[str]:
    west, south, east, north = bbox
    lat0 = int(math.floor(south / 3.0) * 3)
    lat1 = int(math.floor((north - 1e-9) / 3.0) * 3)
    lon0 = int(math.floor(west / 3.0) * 3)
    lon1 = int(math.floor((east - 1e-9) / 3.0) * 3)
    urls = []
    for lat in range(lat0, lat1 + 1, 3):
        ns = "N" if lat >= 0 else "S"
        for lon in range(lon0, lon1 + 1, 3):
            ew = "E" if lon >= 0 else "W"
            tile = f"{ns}{abs(lat):02d}{ew}{abs(lon):03d}"
            urls.append(
                f"https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
                f"ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
            )
    return urls


def fetch_esa_worldcover_labels(
    bbox: tuple[float, float, float, float],
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> tuple[xr.DataArray, xr.DataArray] | tuple[None, None]:
    urls = worldcover_tiles_for_bbox(bbox)
    print(f"ESA WorldCover v200: streaming {len(urls)} COG tile(s) in AOI window...")
    parts = []
    for url in urls:
        try:
            da = _clip_remote_cog(
                url, aoi_wgs84, target_grid, resampling=Resampling.nearest, masked=False
            )
            if da is not None:
                parts.append(da)
        except Exception as exc:
            print(f"  [WARN] Could not load WorldCover tile {url}: {exc}")

    if not parts:
        return None, None

    raw_wc = mosaic_aligned(parts, mode="first").values.astype(np.int32)
    mapped = np.full(raw_wc.shape, IGNORE_LABEL, dtype=np.uint8)
    for wc_code, cls_id in WORLDCOVER_TO_CLASS.items():
        mapped[raw_wc == wc_code] = np.uint8(cls_id)

    # Check 3x3 label homogeneity at target grid resolution
    valid_lbl = mapped != IGNORE_LABEL
    min_nb = minimum_filter(mapped.astype(np.int16), size=3, mode="nearest")
    max_nb = maximum_filter(mapped.astype(np.int16), size=3, mode="nearest")
    is_homogeneous = valid_lbl & (min_nb == max_nb)

    lbl_da = target_grid.copy(data=mapped).astype("uint8")
    hom_da = target_grid.copy(data=is_homogeneous.astype(np.uint8)).astype("uint8")
    return lbl_da, hom_da


def fetch_dswx_hls_cube(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None

    granule_entries = []
    wtr_items = []

    for g in df["granule"].tolist():
        dt = _extract_granule_date(g)
        wtr_url = None
        conf_url = None
        for u in g.data_links(access="external"):
            resolved = _resolve_earthdata_url(g, u)
            if resolved.endswith("_WTR.tif") and "_BWTR.tif" not in resolved:
                wtr_url = resolved
            elif HIGH_CONFIDENCE_DSWX_ONLY and resolved.endswith("_CONF.tif"):
                conf_url = resolved
        if wtr_url is None or (HIGH_CONFIDENCE_DSWX_ONLY and conf_url is None):
            continue
        idx = len(granule_entries)
        granule_entries.append({"date": dt, "conf_url": conf_url})
        wtr_items.append((idx, wtr_url, Resampling.nearest, False))

    if not wtr_items:
        return None

    print(f"OPERA DSWx-HLS (validated): streaming {len(wtr_items)} WTR COG files...")
    wtr_clipped = _stream_and_clip_cogs(
        wtr_items, aoi_wgs84, target_grid, chunk_size=128, max_workers=8, show_progress=True
    )

    conf_items = []
    if HIGH_CONFIDENCE_DSWX_ONLY:
        for idx, wtr in wtr_clipped.items():
            if bool(wtr.isin([1, 2]).any()):
                conf_items.append((idx, granule_entries[idx]["conf_url"], Resampling.nearest, False))
        if conf_items:
            print(f"OPERA DSWx-HLS (validated): streaming {len(conf_items)} CONF COG files (water-bearing scenes)...")
            conf_clipped = _stream_and_clip_cogs(
                conf_items, aoi_wgs84, target_grid, chunk_size=128, max_workers=8, show_progress=True
            )
        else:
            conf_clipped = {}
    else:
        conf_clipped = {}

    by_date_wtr: dict[str, list[xr.DataArray]] = defaultdict(list)
    for idx in sorted(wtr_clipped.keys()):
        wtr = wtr_clipped.pop(idx)
        clear = wtr.isin([0, 1, 2])
        if not bool(clear.any()):
            continue
        if HIGH_CONFIDENCE_DSWX_ONLY and bool(wtr.isin([1, 2]).any()):
            conf = conf_clipped.pop(idx, None)
            if conf is None:
                continue
            # DSWx CONF: 1=open water high conf, 3=partial water high conf; keep clear non-water (wtr==0)
            clear = clear & ((wtr == 0) | conf.isin([1, 3]))
        if not bool(clear.any()):
            continue
        wtr_clean = wtr.where(clear, np.nan).astype("float32")
        by_date_wtr[granule_entries[idx]["date"]].append(wtr_clean)

    if not by_date_wtr:
        return None
    dates = sorted(by_date_wtr.keys())
    per_date = []
    for dt in dates:
        merged = mosaic_aligned(by_date_wtr.pop(dt), mode="first").values
        encoded = np.where(np.isfinite(merged), merged, 255).astype(np.uint8)
        per_date.append(
            xr.DataArray(encoded, coords={"y": target_grid["y"], "x": target_grid["x"]}, dims=("y", "x"))
        )
    da_out = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))
    da_out.attrs["description"] = "0=not_water, 1=open_water, 2=partial_water, 255=invalid_or_cloudy"
    return xr.Dataset({"dswx_hls_wtr": da_out}).rio.write_crs(target_grid.rio.crs)


def rasterize_custom_geojson_labels(
    geojson_path: str | Path,
    target_grid: xr.DataArray,
) -> xr.DataArray | None:
    if geojson_path is None:
        return None
    gdf = gpd.read_file(geojson_path)
    if "class_id" not in gdf.columns:
        raise ValueError("VALIDATED_LABELS_GEOJSON must contain an integer 'class_id' column (0..5).")
    gdf = gdf.to_crs(target_grid.rio.crs)
    shapes = [(geom, int(cid)) for geom, cid in zip(gdf.geometry, gdf["class_id"]) if geom is not None]
    if not shapes:
        return None
    transform = target_grid.rio.transform()
    arr = rasterio.features.rasterize(
        shapes,
        out_shape=target_grid.shape,
        transform=transform,
        fill=IGNORE_LABEL,
        dtype=np.uint8,
    )
    return target_grid.copy(data=arr).astype("uint8")


def fuse_labels_for_timestamp(
    target_date: pd.Timestamp,
    grid_shape: tuple[int, int],
    wc_lbl: np.ndarray | None,
    wc_hom: np.ndarray | None,
    dswx_ds: xr.Dataset | None,
    custom_lbl: np.ndarray | None,
    slope_arr: np.ndarray | None,
) -> tuple[np.ndarray, str | None]:
    if wc_lbl is not None:
        lbl = wc_lbl.copy()
        if REQUIRE_HOMOGENEOUS_3X3 and wc_hom is not None:
            lbl[wc_hom == 0] = IGNORE_LABEL
    else:
        lbl = np.full(grid_shape, IGNORE_LABEL, dtype=np.uint8)

    matched_dswx_date = None
    if dswx_ds is not None and len(dswx_ds["time"]) > 0:
        dswx_times = pd.to_datetime(dswx_ds["time"].values)
        diffs = np.abs((dswx_times - target_date).days)
        best_i = int(np.argmin(diffs))
        if int(diffs[best_i]) <= DSWX_MAX_TIME_DIFF_DAYS:
            matched_dswx_date = str(dswx_times[best_i].date())
            wtr = dswx_ds["dswx_hls_wtr"].isel(time=best_i).values
            # Only apply dynamic optical water overrides on flatter terrain (avoid dark alpine shadow false positives)
            flat_enough = (slope_arr <= MAX_WATER_SLOPE_DEG) if slope_arr is not None else True
            # Open water override -> class 1
            lbl[(wtr == 1.0) & flat_enough] = 1
            # Partial surface water / inundated vegetation over cropland/forest/bare -> class 3 (Wetland)
            inundated_veg = (wtr == 2.0) & flat_enough & np.isin(lbl, [0, 2, 3, 4])
            lbl[inundated_veg] = 3
            # Clear non-water over static permanent water -> mark uncertain (255) to avoid stale labels
            stale_water = (wtr == 0.0) & (lbl == 1)
            lbl[stale_water] = IGNORE_LABEL

    # Custom validated polygons take highest precedence
    if custom_lbl is not None:
        has_custom = custom_lbl != IGNORE_LABEL
        lbl[has_custom] = custom_lbl[has_custom]

    # Mask extreme cliffs / invalid DEM slopes
    if slope_arr is not None:
        bad_slope = ~np.isfinite(slope_arr)
        if MAX_SLOPE_DEG is not None:
            bad_slope = bad_slope | (slope_arr > MAX_SLOPE_DEG)
        lbl[bad_slope] = IGNORE_LABEL

    return lbl, matched_dswx_date


## 6. Execute Acquisition, Build Spatially-Blocked Features & Export Training Sets

For each AOI:
1. Streams & aligns validated cubes (`nasadem_static`, `opera_rtc_s1`, `nisar_gcov`, `nisar_gunw`, `esa_worldcover`, `opera_dswx_hls`) to the AOI UTM grid and saves compressed NetCDF-4 files under `data/training/<RUN_NAME>/<aoi>/`.
2. Builds `3x3` texture-augmented SAR + terrain feature vectors **only where the full `3x3` neighborhood is free of masked/NaN pixels** (`binary_dilation` guard).
3. Assigns deterministic `BLOCK_SIZE_M` spatial blocks to `train` / `val` / `test` (`SPLIT_RATIOS`) across all timestamps to prevent spatial leakage.
4. Exports class-balanced `.npz` & `.parquet` datasets per sensor (`opera_rtc_s1` and/or `nisar_gcov`) plus `dataset_metadata.json` and `validation_manifest.csv`.


In [ ]:
def _get_split_cap(cap_cfg: dict[str, int | None] | int | None, sp_id: int) -> int | None:
    if cap_cfg is None or isinstance(cap_cfg, int):
        return cap_cfg
    return cap_cfg.get(SPLIT_ID_TO_NAME[sp_id])


def assign_spatial_block_splits(
    aoi_name: str,
    xs: np.ndarray,
    ys: np.ndarray,
    block_size_m: float,
    split_ratios: tuple[float, float, float],
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (split_grid [0=train,1=val,2=test], block_id_grid) for a 2D UTM grid."""
    bx = np.floor(xs / block_size_m).astype(np.int64)[None, :]
    by = np.floor(ys / block_size_m).astype(np.int64)[:, None]
    bx_2d = np.broadcast_to(bx, (len(ys), len(xs)))
    by_2d = np.broadcast_to(by, (len(ys), len(xs)))

    t_thr = split_ratios[0]
    v_thr = split_ratios[0] + split_ratios[1]

    split_grid = np.empty((len(ys), len(xs)), dtype=np.uint8)
    block_id_grid = np.empty((len(ys), len(xs)), dtype=object)

    for ux in np.unique(bx):
        x_mask = bx_2d[0, :] == ux
        for uy in np.unique(by):
            y_mask = by_2d[:, 0] == uy
            digest = hashlib.sha256(f"{seed}:{aoi_name}:{int(ux)}:{int(uy)}".encode("utf-8")).digest()
            u01 = int.from_bytes(digest[:8], "big") / float(1 << 64)
            sp = 0 if u01 < t_thr else (1 if u01 < v_thr else 2)
            split_grid[np.ix_(y_mask, x_mask)] = sp
            block_id_grid[np.ix_(y_mask, x_mask)] = f"{aoi_name}_{int(ux)}_{int(uy)}"
    return split_grid, block_id_grid


def extract_3x3_texture_features(
    band_arrays: dict[str, np.ndarray],
    static_arrays: dict[str, np.ndarray],
) -> tuple[np.ndarray, list[str], np.ndarray]:
    """
    Build feature array (H, W, F) with raw dB, 3x3 mean, 3x3 std, and static terrain.
    Returns (features_hwf, feature_names, valid_3x3_mask_hw).
    """
    valid_2d = np.ones(next(iter(band_arrays.values())).shape, dtype=bool)
    for arr in band_arrays.values():
        valid_2d &= np.isfinite(arr)
    for arr in static_arrays.values():
        valid_2d &= np.isfinite(arr)

    # Ensure 3x3 neighborhood has zero NaNs/masked pixels before computing uniform_filter
    valid_3x3 = ~binary_dilation(~valid_2d, structure=np.ones((3, 3), dtype=bool))

    feat_channels = []
    feat_names = []
    for name, arr in band_arrays.items():
        clean = np.where(valid_2d, arr, 0.0).astype(np.float32)
        feat_channels.append(clean)
        feat_names.append(name)
        if name.endswith("_dB") and "minus" not in name:
            mu = uniform_filter(clean, size=3, mode="nearest")
            sq = uniform_filter(clean * clean, size=3, mode="nearest")
            sd = np.sqrt(np.maximum(sq - mu * mu, 0.0)).astype(np.float32)
            feat_channels.extend([mu.astype(np.float32), sd])
            feat_names.extend([f"{name}_mean3", f"{name}_std3"])

    for name, arr in static_arrays.items():
        clean = np.where(valid_2d, arr, 0.0).astype(np.float32)
        feat_channels.append(clean)
        feat_names.append(name)

    stacked = np.stack(feat_channels, axis=-1).astype(np.float32)
    return stacked, feat_names, valid_3x3


def collect_sensor_samples(
    aoi_name: str,
    sensor_name: str,
    sensor_ds: xr.Dataset,
    static_ds: xr.Dataset | None,
    gunw_ds: xr.Dataset | None,
    wc_lbl: np.ndarray | None,
    wc_hom: np.ndarray | None,
    dswx_ds: xr.Dataset | None,
    custom_lbl: np.ndarray | None,
    split_grid: np.ndarray,
    block_id_grid: np.ndarray,
) -> tuple[list[pd.DataFrame], xr.Dataset | None, xr.DataArray | None, list[dict]]:
    static_arrays = {}
    slope_arr = None
    if static_ds is not None:
        for k in ("elevation_m", "slope_deg", "aspect_sin", "aspect_cos"):
            if k in static_ds:
                static_arrays[k] = static_ds[k].values.astype(np.float32)
        slope_arr = static_arrays.get("slope_deg")

    xs_1d = sensor_ds["x"].values
    ys_1d = sensor_ds["y"].values

    if sensor_name == "nisar_gcov" and "HHHH_dB" in sensor_ds and "HVHV_dB" in sensor_ds:
        sar_vars = [v for v in ("HHHH_dB", "HVHV_dB", "HH_minus_HV_dB") if v in sensor_ds]
    elif sensor_name == "opera_rtc_s1" and "VV_dB" in sensor_ds and "VH_dB" in sensor_ds:
        sar_vars = [v for v in ("VV_dB", "VH_dB", "VH_minus_VV_dB") if v in sensor_ds]
    else:
        sar_vars = list(sensor_ds.data_vars)

    frames = []
    label_slices = []
    kept_indices = []
    manifest_rows = []
    feature_names = None

    for t_idx, t_val in enumerate(sensor_ds["time"].values):
        t_ts = pd.Timestamp(t_val)
        date_str = str(t_ts.date())
        band_arrays = {
            var: sensor_ds[var].isel(time=t_idx).values.astype(np.float32)
            for var in sar_vars
        }

        feat_hwf, fnames, valid_3x3 = extract_3x3_texture_features(band_arrays, static_arrays)

        # Attach optional NISAR GUNW coherence + availability flag without invalidating valid GCOV pixels
        if sensor_name == "nisar_gcov" and INCLUDE_NISAR_GUNW:
            raw_coh = np.full(split_grid.shape, np.nan, dtype=np.float32)
            if gunw_ds is not None and "coherence_HH" in gunw_ds and len(gunw_ds["time"]) > 0:
                g_times = pd.to_datetime(gunw_ds["time"].values)
                diffs = np.abs((g_times - t_ts).days)
                bi = int(np.argmin(diffs))
                if int(diffs[bi]) <= GUNW_MAX_TIME_DIFF_DAYS:
                    raw_coh = gunw_ds["coherence_HH"].isel(time=bi).values.astype(np.float32)
            coh_ok = np.isfinite(raw_coh) & (raw_coh >= 0.0) & (raw_coh <= 1.0) & valid_3x3
            coh_2d = np.where(coh_ok, np.clip(raw_coh, 0.0, 1.0), 0.0).astype(np.float32)
            has_gunw_2d = coh_ok.astype(np.float32)
            n_static = len(static_arrays)
            if n_static > 0:
                sar_part = feat_hwf[..., : feat_hwf.shape[-1] - n_static]
                stat_part = feat_hwf[..., feat_hwf.shape[-1] - n_static :]
                feat_hwf = np.concatenate(
                    [sar_part, np.stack([coh_2d, has_gunw_2d], axis=-1), stat_part], axis=-1
                )
                fnames = fnames[:-n_static] + ["coherence_HH", "has_gunw"] + fnames[-n_static:]
            else:
                feat_hwf = np.concatenate([feat_hwf, np.stack([coh_2d, has_gunw_2d], axis=-1)], axis=-1)
                fnames = fnames + ["coherence_HH", "has_gunw"]

        if feature_names is None:
            feature_names = fnames

        lbl_2d, matched_dswx = fuse_labels_for_timestamp(
            t_ts, split_grid.shape, wc_lbl, wc_hom, dswx_ds, custom_lbl, slope_arr
        )
        # Mask non-3x3-valid SAR boundary/layover/shadow pixels in labels_*.nc to prevent SAR-NaN/label mismatch
        lbl_2d = np.where(valid_3x3, lbl_2d, IGNORE_LABEL).astype(np.uint8)
        sample_mask = valid_3x3 & (lbl_2d != IGNORE_LABEL)
        valid_pct = float(valid_3x3.mean() * 100.0)
        labeled_px = int(sample_mask.sum())
        keep_ts = (valid_pct >= MIN_VALID_PCT) and (labeled_px > 0)

        manifest_rows.append(
            {
                "aoi": aoi_name,
                "sensor": sensor_name,
                "date": date_str,
                "dswx_matched_date": matched_dswx or "none",
                "valid_3x3_pct": round(valid_pct, 2),
                "labeled_pixels": labeled_px,
                "kept_in_cube": bool(keep_ts),
            }
        )

        if not keep_ts:
            continue

        label_da = xr.DataArray(lbl_2d, coords={"y": sensor_ds["y"], "x": sensor_ds["x"]}, dims=("y", "x"))
        label_slices.append(label_da)
        kept_indices.append(t_idx)

        # Pre-sample per (split, class) before DataFrame construction to keep 10m RAM low
        ts_seed = int.from_bytes(
            hashlib.sha256(f"{RANDOM_SEED}:{aoi_name}:{sensor_name}:{date_str}".encode("utf-8")).digest()[:8],
            "big",
        )
        rng_ts = np.random.default_rng(ts_seed)
        valid_flat = np.flatnonzero(sample_mask)
        lbl_flat = lbl_2d.ravel()[valid_flat]
        sp_flat = split_grid.ravel()[valid_flat]

        if MAX_SAMPLES_PER_CLASS_PER_TIMESTAMP is not None:
            chosen_parts = []
            for sp_id in (0, 1, 2):
                sp_sel = sp_flat == sp_id
                if not np.any(sp_sel):
                    continue
                ts_cap = _get_split_cap(MAX_SAMPLES_PER_CLASS_PER_TIMESTAMP, sp_id)
                for cls_id in np.unique(lbl_flat[sp_sel]):
                    pool = valid_flat[sp_sel & (lbl_flat == cls_id)]
                    if ts_cap is not None and len(pool) > ts_cap:
                        chosen_parts.append(rng_ts.choice(pool, size=ts_cap, replace=False))
                    else:
                        chosen_parts.append(pool)
            chosen_idx = np.concatenate(chosen_parts) if chosen_parts else np.empty(0, dtype=np.int64)
        else:
            chosen_idx = valid_flat

        if len(chosen_idx) == 0:
            continue

        rows_idx, cols_idx = np.unravel_index(chosen_idx, split_grid.shape)
        feat_flat = feat_hwf.reshape(-1, feat_hwf.shape[-1])
        df_t = pd.DataFrame(feat_flat[chosen_idx], columns=fnames)
        df_t["label"] = lbl_2d.ravel()[chosen_idx].astype(np.int64)
        df_t["split"] = split_grid.ravel()[chosen_idx].astype(np.uint8)
        df_t["block_id"] = block_id_grid.ravel()[chosen_idx]
        df_t["aoi"] = aoi_name
        df_t["date"] = date_str
        df_t["x_utm"] = xs_1d[cols_idx]
        df_t["y_utm"] = ys_1d[rows_idx]
        frames.append(df_t)

    if not kept_indices:
        return frames, None, None, manifest_rows

    valid_sensor_ds = sensor_ds.isel(time=kept_indices)
    kept_times = valid_sensor_ds["time"].values
    labels_cube = xr.concat(
        label_slices, dim=pd.Index(pd.to_datetime(kept_times), name="time")
    )
    if sensor_ds.rio.crs is not None:
        valid_sensor_ds = valid_sensor_ds.rio.write_crs(sensor_ds.rio.crs)
        labels_cube = labels_cube.rio.write_crs(sensor_ds.rio.crs)
    return frames, valid_sensor_ds, labels_cube, manifest_rows


def _sample_balanced_by_aoi(
    grp: pd.DataFrame,
    sp_cap: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    """Water-fill sample quota evenly across AOIs within a (split, class) pool."""
    if len(grp) <= sp_cap:
        return grp
    if "aoi" not in grp.columns or grp["aoi"].nunique() <= 1:
        idx = rng.choice(grp.index.to_numpy(), size=sp_cap, replace=False)
        return grp.loc[idx]

    aoi_pools = {aoi: sub.index.to_numpy() for aoi, sub in grp.groupby("aoi", sort=True, observed=True)}
    ordered_aois = sorted(aoi_pools.keys(), key=lambda k: len(aoi_pools[k]))
    chosen_idx = []
    remaining_budget = sp_cap
    for i, aoi in enumerate(ordered_aois):
        pool = aoi_pools[aoi]
        rem_aois = len(ordered_aois) - i
        quota = (remaining_budget + rem_aois - 1) // rem_aois
        if len(pool) <= quota:
            chosen_idx.append(pool)
            remaining_budget -= len(pool)
        else:
            chosen_idx.append(rng.choice(pool, size=quota, replace=False))
            remaining_budget -= quota
    return grp.loc[np.concatenate(chosen_idx)]


def stratified_sample_and_export(
    sensor_name: str,
    df_all: pd.DataFrame,
    out_dir: Path,
) -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    meta_cols = {"label", "split", "block_id", "aoi", "date", "x_utm", "y_utm"}
    feature_cols = [c for c in df_all.columns if c not in meta_cols]

    # Drop rows missing any polarization/channel across mixed-mode acquisitions (e.g. single-pol HH-only pass)
    finite_mask = np.ones(len(df_all), dtype=bool)
    for col in feature_cols:
        finite_mask &= np.isfinite(df_all[col].to_numpy())
    if not finite_mask.all():
        dropped = int((~finite_mask).sum())
        print(f"  [WARN] {sensor_name}: dropping {dropped:,} row(s) with non-finite/missing schema features.")
        df_all = df_all.loc[finite_mask].reset_index(drop=True)
    del finite_mask

    sampled_parts = []
    for sp_id in (0, 1, 2):
        df_sp = df_all[df_all["split"] == sp_id]
        sp_cap = _get_split_cap(MAX_SAMPLES_PER_CLASS_PER_SPLIT, sp_id)
        for cls_id, grp in df_sp.groupby("label", observed=True):
            if sp_cap is not None and len(grp) > sp_cap:
                sampled_parts.append(_sample_balanced_by_aoi(grp, sp_cap, rng))
            else:
                sampled_parts.append(grp)

    balanced_df = pd.concat(sampled_parts, ignore_index=True)
    for col in ("block_id", "aoi", "date"):
        if col in balanced_df.columns:
            balanced_df[col] = balanced_df[col].astype(str)
    parquet_path = out_dir / f"training_samples_{sensor_name}.parquet"
    balanced_df.to_parquet(parquet_path, index=False)

    split_arrays = {}
    counts_summary = {}
    aoi_summary = {}
    for sp_id, sp_name in SPLIT_ID_TO_NAME.items():
        sub = balanced_df[balanced_df["split"] == sp_id]
        split_arrays[f"X_{sp_name}"] = sub[feature_cols].to_numpy(dtype=np.float32)
        split_arrays[f"y_{sp_name}"] = sub["label"].to_numpy(dtype=np.int64)
        counts_summary[sp_name] = {int(k): int(v) for k, v in sub["label"].value_counts().sort_index().items()}
        aoi_summary[sp_name] = {str(k): int(v) for k, v in sub["aoi"].value_counts().sort_index().items()}

    npz_path = out_dir / f"training_samples_{sensor_name}.npz"
    np.savez_compressed(
        npz_path,
        **split_arrays,
        feature_names=np.array(feature_cols),
        class_ids=np.array(sorted(CLASS_DEFINITIONS.keys()), dtype=np.int64),
    )
    print(f"[EXPORT] {sensor_name}: {len(balanced_df):,} balanced samples -> {npz_path.name} & {parquet_path.name}")
    return {
        "feature_cols": feature_cols,
        "counts": counts_summary,
        "aoi_counts": aoi_summary,
        "total_rows": len(balanced_df),
    }


def _save_qa_slice(
    qa_artifacts: dict,
    aoi_name: str,
    s_name: str,
    s_ds: xr.Dataset,
    lbl_cube: xr.DataArray,
    split_grid: np.ndarray,
    cache_dir: Path | None = None,
) -> None:
    if qa_artifacts:
        return
    lbl_counts = [(lbl_cube.isel(time=i).values != IGNORE_LABEL).sum() for i in range(len(lbl_cube["time"]))]
    if not lbl_counts or max(lbl_counts) == 0:
        return
    best_i = int(np.argmax(lbl_counts))
    s_slice = s_ds.isel(time=[best_i]).load()
    l_slice = lbl_cube.isel(time=[best_i]).load()
    qa_artifacts[f"{aoi_name}:{s_name}"] = {
        "sensor_ds": s_slice,
        "labels_cube": l_slice,
        "split_grid": split_grid.copy(),
    }
    if cache_dir is not None:
        s_slice.to_netcdf(cache_dir / "_qa_sensor.nc", engine="h5netcdf")
        xr.Dataset({"label": l_slice}).to_netcdf(cache_dir / "_qa_labels.nc", engine="h5netcdf")
        np.save(cache_dir / "_qa_split.npy", split_grid)
        (cache_dir / "_qa_meta.json").write_text(json.dumps({"key": f"{aoi_name}:{s_name}"}), encoding="utf-8")


def _load_concat_cached_parquets(parquet_paths: list[Path]) -> pd.DataFrame:
    cat_cols = ("block_id", "aoi", "date")
    uniques = {c: set() for c in cat_cols}
    for p in parquet_paths:
        meta_df = pd.read_parquet(p, columns=list(cat_cols))
        for c in cat_cols:
            uniques[c].update(meta_df[c].astype(str).unique())
    cat_dtypes = {c: pd.CategoricalDtype(sorted(uniques[c])) for c in cat_cols}
    parts = []
    for p in parquet_paths:
        part = pd.read_parquet(p)
        for c, dt in cat_dtypes.items():
            if c in part.columns:
                part[c] = part[c].astype(str).astype(dt)
        parts.append(part)
    combined = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()
    return combined


def _write_cf_netcdf(ds_in: xr.Dataset, out_path: Path, crs) -> None:
    ds_cf = ds_in.copy()
    if "spatial_ref" in ds_cf.variables:
        ds_cf = ds_cf.drop_vars("spatial_ref")
    ds_cf = ds_cf.rio.write_crs(crs).rio.write_coordinate_system()
    for v in ds_cf.data_vars:
        ds_cf[v].attrs.pop("grid_mapping", None)
        ds_cf[v].encoding["grid_mapping"] = "spatial_ref"
        ds_cf[v].encoding["coordinates"] = "spatial_ref"
    enc = {v: {"zlib": True, "complevel": 4} for v in ds_cf.data_vars}
    ds_cf.to_netcdf(out_path, engine="h5netcdf", encoding=enc)


BATCH_SAR_DATES = None if MAX_TIMESTEPS_PER_AOI is not None else 25
USE_AOI_PARQUET_CACHE = RUN_FETCH and (MAX_TIMESTEPS_PER_AOI is None)
CACHE_DIR = OUT_ROOT / "_aoi_cache"
if USE_AOI_PARQUET_CACHE:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

sensor_frames: dict[str, list[pd.DataFrame]] = defaultdict(list)
cached_parquets_by_sensor: dict[str, list[Path]] = defaultdict(list)
all_manifest_rows: list[dict] = []
qa_artifacts: dict[str, dict] = {}

for aoi_name, bbox in TRAINING_AOIS.items():
    aoi_t0 = time.perf_counter()
    aoi_out = OUT_ROOT / aoi_name
    sel_df = selected_by_aoi.get(aoi_name, pd.DataFrame())

    def _has_ds(ds_key: str) -> bool:
        return (not sel_df.empty) and (not sel_df[sel_df["dataset"] == ds_key].empty)

    req_cubes = []
    if SAVE_INTERMEDIATE_CUBES:
        if INCLUDE_NASADEM and _has_ds("nasadem"):
            req_cubes.append("nasadem_static.nc")
        if INCLUDE_OPERA_RTC_S1 and _has_ds("opera_rtc_s1"):
            req_cubes.extend(["opera_rtc_s1.nc", "labels_opera_rtc_s1.nc"])
        if INCLUDE_NISAR_GCOV and _has_ds("nisar_gcov"):
            req_cubes.extend(["nisar_gcov.nc", "labels_nisar_gcov.nc"])
        if INCLUDE_NISAR_GUNW and _has_ds("nisar_gunw"):
            req_cubes.append("nisar_gunw.nc")
        if INCLUDE_DSWX_HLS and _has_ds("opera_dswx_hls"):
            req_cubes.append("opera_dswx_hls.nc")
        if INCLUDE_ESA_WORLDCOVER:
            req_cubes.append("worldcover_static.nc")
    cubes_ready = (not SAVE_INTERMEDIATE_CUBES) or all((aoi_out / f).exists() for f in req_cubes)

    if USE_AOI_PARQUET_CACHE and (CACHE_DIR / f"{aoi_name}.done").exists() and cubes_ready:
        print(f"\n[CACHE HIT] Loading completed AOI '{aoi_name}' from {CACHE_DIR}")
        m_path = CACHE_DIR / f"{aoi_name}__manifest.json"
        if m_path.exists():
            all_manifest_rows.extend(json.loads(m_path.read_text(encoding="utf-8")))
        for s_name in ("opera_rtc_s1", "nisar_gcov"):
            pq_p = CACHE_DIR / f"{aoi_name}__{s_name}.parquet"
            if pq_p.exists():
                cached_parquets_by_sensor[s_name].append(pq_p)
        qa_meta_p = CACHE_DIR / "_qa_meta.json"
        if not qa_artifacts and qa_meta_p.exists():
            qa_key = json.loads(qa_meta_p.read_text(encoding="utf-8"))["key"]
            with xr.open_dataset(CACHE_DIR / "_qa_sensor.nc", engine="h5netcdf") as s_in:
                s_qa = s_in.load()
            with xr.open_dataset(CACHE_DIR / "_qa_labels.nc", engine="h5netcdf") as l_in:
                l_qa = l_in["label"].load()
            qa_artifacts[qa_key] = {
                "sensor_ds": s_qa,
                "labels_cube": l_qa,
                "split_grid": np.load(CACHE_DIR / "_qa_split.npy"),
            }
        continue

    aoi_gdf = gpd.GeoDataFrame(geometry=[box(*bbox)], crs="EPSG:4326")
    utm_crs = aoi_gdf.estimate_utm_crs()
    target_grid = build_target_grid(aoi_gdf, utm_crs, TARGET_RES_M)
    if SAVE_INTERMEDIATE_CUBES:
        aoi_out.mkdir(parents=True, exist_ok=True)

    print(f"\n================ Processing AOI: {aoi_name} ({utm_crs}) ================")
    if RUN_FETCH:
        nasadem_ds = (
            fetch_nasadem_static(aoi_name, sel_df[sel_df["dataset"] == "nasadem"], aoi_gdf, target_grid)
            if INCLUDE_NASADEM else None
        )
        gcov_ds = (
            fetch_nisar_gcov_validated(sel_df[sel_df["dataset"] == "nisar_gcov"], aoi_gdf, target_grid)
            if INCLUDE_NISAR_GCOV else None
        )
        gunw_ds = (
            fetch_nisar_gunw_validated(sel_df[sel_df["dataset"] == "nisar_gunw"], aoi_gdf, target_grid)
            if INCLUDE_NISAR_GUNW else None
        )
        wc_lbl_da, wc_hom_da = (
            fetch_esa_worldcover_labels(bbox, aoi_gdf, target_grid)
            if INCLUDE_ESA_WORLDCOVER else (None, None)
        )
        dswx_ds = (
            fetch_dswx_hls_cube(sel_df[sel_df["dataset"] == "opera_dswx_hls"], aoi_gdf, target_grid)
            if INCLUDE_DSWX_HLS else None
        )
        custom_lbl_da = rasterize_custom_geojson_labels(VALIDATED_LABELS_GEOJSON, target_grid)

        if SAVE_INTERMEDIATE_CUBES:
            for cname, cds in (
                ("nasadem_static", nasadem_ds),
                ("nisar_gunw", gunw_ds),
                ("opera_dswx_hls", dswx_ds),
            ):
                if cds is not None:
                    _write_cf_netcdf(cds, aoi_out / f"{cname}.nc", target_grid.rio.crs)
            if wc_lbl_da is not None:
                wc_ds = xr.Dataset({"worldcover_class": wc_lbl_da, "is_homogeneous_3x3": wc_hom_da})
                _write_cf_netcdf(wc_ds, aoi_out / "worldcover_static.nc", target_grid.rio.crs)
        rtc_ds = (
            fetch_opera_rtc_validated(sel_df[sel_df["dataset"] == "opera_rtc_s1"], aoi_gdf, target_grid)
            if (BATCH_SAR_DATES is None and INCLUDE_OPERA_RTC_S1) else None
        )
    else:
        def _load_opt(fname):
            p = aoi_out / fname
            if not p.exists():
                return None
            with xr.open_dataset(p, engine="h5netcdf") as ds_in:
                return ds_in.load()

        nasadem_ds = _load_opt("nasadem_static.nc")
        rtc_ds = _load_opt("opera_rtc_s1.nc")
        gcov_ds = _load_opt("nisar_gcov.nc")
        gunw_ds = _load_opt("nisar_gunw.nc")
        dswx_ds = _load_opt("opera_dswx_hls.nc")
        wc_ds = _load_opt("worldcover_static.nc")
        wc_lbl_da = wc_ds["worldcover_class"] if wc_ds is not None else None
        wc_hom_da = wc_ds["is_homogeneous_3x3"] if wc_ds is not None else None
        custom_lbl_da = rasterize_custom_geojson_labels(VALIDATED_LABELS_GEOJSON, target_grid)

    wc_lbl_arr = wc_lbl_da.values if wc_lbl_da is not None else None
    wc_hom_arr = wc_hom_da.values if wc_hom_da is not None else None
    custom_arr = custom_lbl_da.values if custom_lbl_da is not None else None

    split_grid, block_id_grid = assign_spatial_block_splits(
        aoi_name, target_grid["x"].values, target_grid["y"].values, BLOCK_SIZE_M, SPLIT_RATIOS, RANDOM_SEED
    )

    aoi_frames: dict[str, list[pd.DataFrame]] = defaultdict(list)
    aoi_mrows: list[dict] = []

    def _write_sensor_cubes(s_name: str, v_ds: xr.Dataset, l_cube: xr.DataArray) -> None:
        _write_cf_netcdf(v_ds, aoi_out / f"{s_name}.nc", target_grid.rio.crs)
        _write_cf_netcdf(xr.Dataset({"label": l_cube}), aoi_out / f"labels_{s_name}.nc", target_grid.rio.crs)

    def _consume_sensor(
        s_name: str, s_ds: xr.Dataset | None, save_cubes: bool, collect_frames: bool = True
    ) -> tuple[xr.Dataset | None, xr.DataArray | None]:
        if s_ds is None:
            return None, None
        frames, valid_ds, lbl_cube, mrows = collect_sensor_samples(
            aoi_name, s_name, s_ds, nasadem_ds, gunw_ds,
            wc_lbl_arr, wc_hom_arr, dswx_ds, custom_arr, split_grid, block_id_grid,
        )
        if valid_ds is not None and lbl_cube is not None:
            if save_cubes:
                _write_sensor_cubes(s_name, valid_ds, lbl_cube)
            _save_qa_slice(
                qa_artifacts, aoi_name, s_name, valid_ds, lbl_cube, split_grid,
                cache_dir=(CACHE_DIR if USE_AOI_PARQUET_CACHE else None),
            )
        if collect_frames:
            aoi_frames[s_name].extend(frames)
            aoi_mrows.extend(mrows)
        return valid_ds, lbl_cube

    if BATCH_SAR_DATES is None or not RUN_FETCH:
        _consume_sensor("opera_rtc_s1", rtc_ds, SAVE_INTERMEDIATE_CUBES)
        _consume_sensor("nisar_gcov", gcov_ds, SAVE_INTERMEDIATE_CUBES)
    else:
        if INCLUDE_OPERA_RTC_S1:
            rtc_df = sel_df[sel_df["dataset"] == "opera_rtc_s1"]
            rtc_cache_pq = CACHE_DIR / f"{aoi_name}__opera_rtc_s1.parquet"
            rtc_cache_manifest = CACHE_DIR / f"{aoi_name}__manifest.json"
            if USE_AOI_PARQUET_CACHE and rtc_cache_pq.exists():
                print(f"[{aoi_name}] Reusing cached opera_rtc_s1 parquet: {rtc_cache_pq.name}")
                cached_parquets_by_sensor["opera_rtc_s1"].append(rtc_cache_pq)
                rtc_cached_rows = (
                    [r for r in json.loads(rtc_cache_manifest.read_text(encoding="utf-8")) if r.get("sensor") == "opera_rtc_s1"]
                    if rtc_cache_manifest.exists() else []
                )
                rtc_cube_dates: set[str] = set()
                if SAVE_INTERMEDIATE_CUBES:
                    rtc_nc = aoi_out / "opera_rtc_s1.nc"
                    lbl_nc = aoi_out / "labels_opera_rtc_s1.nc"
                    if rtc_nc.exists() and lbl_nc.exists():
                        with xr.open_dataset(rtc_nc, engine="h5netcdf") as ds_in:
                            rtc_cube_dates = {str(pd.Timestamp(t).date()) for t in ds_in["time"].values}
                    else:
                        rtc_cube_df = select_dates_evenly(rtc_df, MAX_CUBE_TIMESTEPS_PER_AOI)
                        print(f"[{aoi_name}] Building opera_rtc_s1 U-TAE cube ({rtc_cube_df['date'].nunique()} dates)...")
                        rtc_cube_ds = fetch_opera_rtc_validated(rtc_cube_df, aoi_gdf, target_grid)
                        v_cube, _ = _consume_sensor("opera_rtc_s1", rtc_cube_ds, True, collect_frames=False)
                        if v_cube is not None:
                            rtc_cube_dates = {str(pd.Timestamp(t).date()) for t in v_cube["time"].values}
                        del rtc_cube_ds, v_cube
                        gc.collect()
                for r in rtc_cached_rows:
                    r["kept_in_cube"] = str(r.get("date")) in rtc_cube_dates
                aoi_mrows.extend(rtc_cached_rows)
            else:
                rtc_dates = sorted(d for d in rtc_df["date"].dropna().unique())
                cube_target_dates = (
                    set(select_dates_evenly(rtc_df, MAX_CUBE_TIMESTEPS_PER_AOI)["date"].astype(str))
                    if SAVE_INTERMEDIATE_CUBES else set()
                )
                rtc_cube_parts = []
                rtc_lbl_parts = []
                for b0 in range(0, len(rtc_dates), BATCH_SAR_DATES):
                    b_dates = rtc_dates[b0 : b0 + BATCH_SAR_DATES]
                    print(f"[{aoi_name}] RTC-S1 batch {b0 // BATCH_SAR_DATES + 1}/{(len(rtc_dates) + BATCH_SAR_DATES - 1) // BATCH_SAR_DATES} ({len(b_dates)} dates)...")
                    rtc_batch = fetch_opera_rtc_validated(rtc_df[rtc_df["date"].isin(b_dates)], aoi_gdf, target_grid)
                    v_batch, l_batch = _consume_sensor("opera_rtc_s1", rtc_batch, False)
                    if SAVE_INTERMEDIATE_CUBES and v_batch is not None and l_batch is not None:
                        keep_idx = [
                            idx for idx, tv in enumerate(v_batch["time"].values)
                            if str(pd.Timestamp(tv).date()) in cube_target_dates
                        ]
                        if keep_idx:
                            rtc_cube_parts.append(v_batch.isel(time=keep_idx).load())
                            rtc_lbl_parts.append(l_batch.isel(time=keep_idx).load())
                    del rtc_batch, v_batch, l_batch
                    gc.collect()
                if SAVE_INTERMEDIATE_CUBES and rtc_cube_parts:
                    full_rtc_cube = xr.concat(rtc_cube_parts, dim="time").sortby("time")
                    full_lbl_cube = xr.concat(rtc_lbl_parts, dim="time").sortby("time")
                    _write_sensor_cubes("opera_rtc_s1", full_rtc_cube, full_lbl_cube)
                    saved_cube_dates = {str(pd.Timestamp(t).date()) for t in full_rtc_cube["time"].values}
                    for r in aoi_mrows:
                        if r.get("sensor") == "opera_rtc_s1":
                            r["kept_in_cube"] = str(r.get("date")) in saved_cube_dates
                    del rtc_cube_parts, rtc_lbl_parts, full_rtc_cube, full_lbl_cube
                    gc.collect()
        _consume_sensor("nisar_gcov", gcov_ds, SAVE_INTERMEDIATE_CUBES)

    all_manifest_rows.extend(aoi_mrows)
    if USE_AOI_PARQUET_CACHE:
        for s_name, dfs in aoi_frames.items():
            if dfs:
                pq_p = CACHE_DIR / f"{aoi_name}__{s_name}.parquet"
                aoi_df = pd.concat(dfs, ignore_index=True)
                aoi_df.to_parquet(pq_p, index=False)
                cached_parquets_by_sensor[s_name].append(pq_p)
                del aoi_df
        (CACHE_DIR / f"{aoi_name}__manifest.json").write_text(json.dumps(aoi_mrows), encoding="utf-8")
        (CACHE_DIR / f"{aoi_name}.done").write_text("ok", encoding="utf-8")
    else:
        for s_name, dfs in aoi_frames.items():
            sensor_frames[s_name].extend(dfs)

    print(f"[AOI DONE] {aoi_name} finished in {(time.perf_counter() - aoi_t0) / 60.0:.2f} min")
    del nasadem_ds, rtc_ds, gcov_ds, gunw_ds, dswx_ds, wc_lbl_da, wc_hom_da, custom_lbl_da, split_grid, block_id_grid, aoi_frames
    gc.collect()

manifest_df = pd.DataFrame(all_manifest_rows)
if not manifest_df.empty:
    manifest_df.to_csv(OUT_ROOT / "validation_manifest.csv", index=False)
    display(manifest_df)

export_summary = {}
all_export_sensors = sorted(set(sensor_frames.keys()) | set(cached_parquets_by_sensor.keys()))
for s_name in all_export_sensors:
    if cached_parquets_by_sensor.get(s_name):
        combined = _load_concat_cached_parquets(cached_parquets_by_sensor[s_name])
    elif sensor_frames.get(s_name):
        combined = pd.concat(sensor_frames[s_name], ignore_index=True)
    else:
        continue
    export_summary[s_name] = stratified_sample_and_export(s_name, combined, OUT_ROOT)
    del combined
    gc.collect()

metadata = {
    "run_profile": RUN_PROFILE,
    "run_name": RUN_NAME,
    "training_aois": list(TRAINING_AOIS.keys()),
    "max_timesteps_per_aoi": MAX_TIMESTEPS_PER_AOI,
    "max_cube_timesteps_per_aoi": MAX_CUBE_TIMESTEPS_PER_AOI,
    "target_res_m": TARGET_RES_M,
    "min_valid_pct": MIN_VALID_PCT,
    "min_bbox_coverage_pct": MIN_BBOX_COVERAGE_PCT,
    "db_clamp_range": [DB_MIN, DB_MAX],
    "allow_provisional_nisar": ALLOW_PROVISIONAL_NISAR,
    "allow_provisional_gunw": ALLOW_PROVISIONAL_GUNW,
    "mask_opera_layover_shadow": MASK_OPERA_LAYOVER_SHADOW,
    "max_slope_deg": MAX_SLOPE_DEG,
    "max_water_slope_deg": MAX_WATER_SLOPE_DEG,
    "require_homogeneous_3x3": REQUIRE_HOMOGENEOUS_3X3,
    "dswx_max_time_diff_days": DSWX_MAX_TIME_DIFF_DAYS,
    "gunw_max_time_diff_days": GUNW_MAX_TIME_DIFF_DAYS,
    "block_size_m": BLOCK_SIZE_M,
    "split_ratios": list(SPLIT_RATIOS),
    "max_samples_per_class_per_timestamp": MAX_SAMPLES_PER_CLASS_PER_TIMESTAMP,
    "max_samples_per_class_per_split": MAX_SAMPLES_PER_CLASS_PER_SPLIT,
    "save_intermediate_cubes": SAVE_INTERMEDIATE_CUBES,
    "classes": {str(k): v[0] for k, v in CLASS_DEFINITIONS.items()},
    "exports": export_summary,
}
(OUT_ROOT / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved metadata ->", OUT_ROOT / "dataset_metadata.json")


## 7. Validation QA & Feature Separation Diagnostics

In [ ]:
if not qa_artifacts:
    print("No sensor datasets exported yet; nothing to plot.")
else:
    first_key = next(iter(qa_artifacts))
    qa = qa_artifacts[first_key]
    s_ds = qa["sensor_ds"]
    lbl_cube = qa["labels_cube"]
    split_grid = qa["split_grid"]

    # Pick timestamp with most labeled pixels
    lbl_counts = [(lbl_cube.isel(time=i).values != IGNORE_LABEL).sum() for i in range(len(lbl_cube["time"]))]
    best_idx = int(np.argmax(lbl_counts))
    best_date = str(pd.Timestamp(lbl_cube["time"].values[best_idx]).date())
    lbl_2d = lbl_cube.isel(time=best_idx).values

    first_var = next(iter(s_ds.data_vars))
    sar_2d = s_ds[first_var].isel(time=best_idx).values

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    # 1. SAR Backscatter (masked)
    axes[0, 0].imshow(sar_2d, cmap="gray", vmin=-25, vmax=0)
    axes[0, 0].set_title(f"{first_key} — {first_var} ({best_date})")

    # 2. Validated Fused Labels
    cmap_colors = [CLASS_DEFINITIONS[i][1] for i in range(6)] + ["#1E293B"]
    cmap = mcolors.ListedColormap(cmap_colors)
    lbl_plot = np.where(lbl_2d == IGNORE_LABEL, 6, lbl_2d)
    im = axes[0, 1].imshow(lbl_plot, cmap=cmap, vmin=0, vmax=6, interpolation="nearest")
    axes[0, 1].set_title("Validated Labels (Dark = Masked/Ignored)")
    cbar = plt.colorbar(im, ax=axes[0, 1], ticks=list(range(7)))
    cbar.ax.set_yticklabels([CLASS_DEFINITIONS[i][0] for i in range(6)] + ["Ignored (255)"])

    # 3. Spatial Block Split
    split_cmap = mcolors.ListedColormap(["#3B82F6", "#F59E0B", "#EF4444"])
    im_sp = axes[1, 0].imshow(split_grid, cmap=split_cmap, vmin=0, vmax=2, interpolation="nearest")
    axes[1, 0].set_title(f"Spatial Block Split ({BLOCK_SIZE_M:.0f}m blocks: Train/Val/Test)")
    cbar_sp = plt.colorbar(im_sp, ax=axes[1, 0], ticks=[0, 1, 2])
    cbar_sp.ax.set_yticklabels(["Train", "Val", "Test"])

    # 4. Feature Separation Scatter (co-pol vs cross-pol)
    sensor_only = first_key.split(":", 1)[1]
    pq_path = OUT_ROOT / f"training_samples_{sensor_only}.parquet"
    if pq_path.exists():
        sample_df = pd.read_parquet(pq_path)
        plot_sub = sample_df.sample(min(len(sample_df), 4000), random_state=RANDOM_SEED)
        x_col = "VV_dB" if "VV_dB" in plot_sub else ("HHHH_dB" if "HHHH_dB" in plot_sub else first_var)
        y_col = "VH_dB" if "VH_dB" in plot_sub else ("HVHV_dB" if "HVHV_dB" in plot_sub else first_var)
        for cid, (cname, c_hex) in CLASS_DEFINITIONS.items():
            sub_c = plot_sub[plot_sub["label"] == cid]
            if not sub_c.empty:
                axes[1, 1].scatter(sub_c[x_col], sub_c[y_col], s=6, alpha=0.45, color=c_hex, label=cname)
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel(y_col)
        axes[1, 1].set_title("Validated Class Backscatter Separation")
        axes[1, 1].legend(markerscale=2, fontsize=8)
    else:
        axes[1, 1].axis("off")

    for ax in axes.ravel()[:3]:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()
